In [20]:
import pypdf
import chromadb
import numpy as np

from sentence_transformers import SentenceTransformer

print("All imports successful ✅")

All imports successful ✅


In [21]:
import pypdf
import chromadb
import numpy as np

from sentence_transformers import SentenceTransformer

print("All imports successful ✅")

All imports successful ✅


In [22]:
from pathlib import Path
from pypdf import PdfReader

pdf_path=Path("../data/machine_learning_100_pages.pdf")
reader=PdfReader(pdf_path)
print("number of pages:",len(reader.pages))

number of pages: 100


In [23]:
first_page=reader.pages[0]
print(first_page)

{'/Contents': IndirectObject(107, 0, 14822615216), '/MediaBox': [0, 0, 612, 792], '/Parent': IndirectObject(106, 0, 14822615216), '/Resources': {'/Font': IndirectObject(1, 0, 14822615216), '/ProcSet': ['/PDF', '/Text', '/ImageB', '/ImageC', '/ImageI']}, '/Rotate': 0, '/Trans': {}, '/Type': '/Page'}


In [24]:
data_folder=Path("../data")
pdf_files=list(data_folder.glob("*.pdf"))
print("PDF files found",len(pdf_files))
for pdf in pdf_files:
    print(pdf.name)

PDF files found 5
machine_learning_100_pages.pdf
04_python_data_science.pdf
01_artificial_intelligence.pdf
02_machine_learning.pdf
03_retrieval_augmented_generation.pdf


In [25]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

In [26]:
documents=[]
for pdf_file in pdf_files:
    print(f"Loading:{pdf_file.name}")
    loader=PyPDFLoader(str(pdf_file))
    docs=loader.load()
    for doc in docs:
        doc.metadata["source_file"]=pdf_file.name
    documents.extend(docs)
print("Total documents loaded:", len(documents))        


Loading:machine_learning_100_pages.pdf
Loading:04_python_data_science.pdf
Loading:01_artificial_intelligence.pdf
Loading:02_machine_learning.pdf
Loading:03_retrieval_augmented_generation.pdf
Total documents loaded: 105


In [27]:
documents[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': 'anonymous',
 'creationdate': '2026-08-17T22:54:58+00:00',
 'author': 'anonymous',
 'keywords': '',
 'moddate': '2026-08-17T22:54:58+00:00',
 'subject': 'unspecified',
 'title': 'untitled',
 'trapped': '/False',
 'source': '../data/machine_learning_100_pages.pdf',
 'total_pages': 100,
 'page': 0,
 'page_label': '1',
 'source_file': 'machine_learning_100_pages.pdf'}

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [29]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len

)
chunks=text_splitter.split_documents(documents)

In [30]:
print("original size",len(documents))
print("chunk size",len(chunks))

original size 105
chunk size 1040


In [31]:
from langchain_huggingface import HuggingFaceEmbeddings

In [32]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
test_embedding = embedding_model.embed_query(
    "What is machine learning?"
)

print("Embedding type:", type(test_embedding))
print("Embedding size:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding type: <class 'list'>
Embedding size: 384
First 10 values: [-0.01995455101132393, 0.00987799558788538, 0.010249645449221134, 0.029553702101111412, 0.027186434715986252, -0.01929652877151966, -0.024129629135131836, -0.037735715508461, -0.04105418920516968, -0.0014749818947166204]


In [33]:
from langchain_chroma import Chroma

In [34]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory= "../chroma_db"
)

In [35]:
query="What is Machine Learning?"
results=vector_db.similarity_search(query,k=3)

In [36]:
for i,doc in enumerate(results,start=1):
    print(f"--RESULT {i}---")
    print("source:",doc.metadata.get("source_file"))
    print("page:",doc.metadata.get("page"))
    print()
    print(doc.page_content[:500])

--RESULT 1---
source: machine_learning_100_pages.pdf
page: 1

The broader lesson is that machine learning is an experimental discipline. Good results come
from forming hypotheses, designing controlled comparisons, inspecting errors, and iterating
carefully. Types of Machine Learning is most useful when it becomes part of that disciplined
process rather than a one-time modeling trick.
Example scenario: imagine forecasting product demand. A chronological split is usually more
--RESULT 2---
source: machine_learning_100_pages.pdf
page: 1

The broader lesson is that machine learning is an experimental discipline. Good results come
from forming hypotheses, designing controlled comparisons, inspecting errors, and iterating
carefully. Types of Machine Learning is most useful when it becomes part of that disciplined
process rather than a one-time modeling trick.
Example scenario: imagine forecasting product demand. A chronological split is usually more
--RESULT 3---
source: machine_learning_100

In [37]:
results_with_scores = vector_db.similarity_search_with_score(
    "What is machine learning?",
    k=3
)

for i, (doc, score) in enumerate(results_with_scores, start=1):
    print(f"\n--- Result {i} ---")
    print("Score:", score)
    print("Source:", doc.metadata.get("source_file"))
    print(doc.page_content[:400])


--- Result 1 ---
Score: 0.5910059809684753
Source: machine_learning_100_pages.pdf
The broader lesson is that machine learning is an experimental discipline. Good results come
from forming hypotheses, designing controlled comparisons, inspecting errors, and iterating
carefully. Types of Machine Learning is most useful when it becomes part of that disciplined
process rather than a one-time modeling trick.
Example scenario: imagine forecasting product demand. A chronological split

--- Result 2 ---
Score: 0.5910059809684753
Source: machine_learning_100_pages.pdf
The broader lesson is that machine learning is an experimental discipline. Good results come
from forming hypotheses, designing controlled comparisons, inspecting errors, and iterating
carefully. Types of Machine Learning is most useful when it becomes part of that disciplined
process rather than a one-time modeling trick.
Example scenario: imagine forecasting product demand. A chronological split

--- Result 3 ---
Score: 0.59100

In [39]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("LLM created successfully")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.